# 8장. 더 빠르게: 실제 LLM처럼 계산하기

5장의 멀티헤드 어텐션은 헤드 4개를 반복문으로 **하나씩** 계산했어요. 이 장에서는 실제 LLM처럼 **헤드 4개를 한 번에** 계산하는 코드를 직접 짜고,
5장의 결과와 똑같은지 확인합니다. 9장부터는 이 방식으로 계산하는 빠른 GPT로 학습해요.

계산을 확인하는 부분(8.2~8.3)은 CPU로도 되지만, **속도를 재는 부분(8.1, 8.4)은 T4 GPU**에서 실행해야 의미가 있습니다. (런타임 유형: T4 GPU)


### 준비
7장의 기본 GPT(`TinyGPT`)와 패키지의 빠른 GPT(`GPT`), 시간을 재는 `time_steps`를 불러오고, 이야기 2,000편을 번호로 바꿔 둡니다.


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import torch
from torch import nn
from torch.nn import functional as F
from modu_llm import Vocab, GPTConfig, Embeddings, MultiHead
from modu_llm import TinyGPT, GPT, copy_basic_to_fast
from modu_llm import load_stories, encode_texts, get_batch, time_steps, get_device

device = get_device()
vocab = Vocab.build()
cfg = GPTConfig()
stories = load_stories(2000)
data = encode_texts(stories, vocab)


## 8.1 우리 코드는 왜 느릴까

7장 GPT로 학습 몇 단계에 걸리는 시간을 재어 봅니다. 느린 이유는 두 가지예요.

1. 헤드 4개를 **반복문으로 하나씩** 계산합니다. GPU는 큰 계산 하나를 한꺼번에 할 때 가장 빨라요.
2. 모든 숫자를 32비트로 계산합니다. T4는 16비트 계산을 훨씬 빠르게 하는 전용 장치를 갖고 있어요.


In [ ]:
basic = TinyGPT(cfg).to(device)
if device == "cuda":
    n = 10
else:
    n = 2
t_basic = time_steps(basic, data, n=n)
print(f"기본 GPT: 한 단계 {t_basic:.0f}ms")


## 8.2 헤드를 한 번에 계산하기

5장과 같은 문장, 같은 크기(벡터 16, 헤드 4개)로 돌아가서, 헤드 4개를 한 번에 계산하는 코드를 직접 짭니다. 순서는 네 걸음이에요.

1. 헤드 4개의 레시피를 위아래로 쌓아 큰 Linear 하나로 만들기
2. `view`와 `transpose`로 헤드를 맨 앞에 세우기
3. 4장의 네 줄(점수 → 마스크 → softmax → 가중합)을 그대로 돌리기
4. 헤드 결과를 이어 붙여 5장 `MultiHead`와 같은지 확인하기


In [ ]:
small = GPTConfig(vocab_size=len(vocab), block_size=16, n_embd=16, n_head=4)
torch.manual_seed(42)
emb = Embeddings(small)
mh = MultiHead(small)

sentence = "토끼가 숲에 가요"
x = emb(torch.tensor(vocab.encode(sentence)))
print("x 모양:", x.shape)
print("헤드 수:", len(mh.heads))


진짜 계산 전에 숫자 16개로 **나누고, 세우고, 되돌리기**를 연습합니다. 숫자가 어디로 가는지 눈으로 따라가 보세요.


In [ ]:
a = torch.arange(16).view(2, 8)      # 글자 2개, 숫자 8개
print(a)
b = a.view(2, 2, 4)                  # 글자마다 헤드 2개 몫으로 나누기
print(b)
c = b.transpose(0, 1)                # 헤드마다 표 하나
print(c)
d = c.transpose(0, 1).reshape(2, 8)  # 되돌리기
print(torch.equal(d, a))


**① 레시피 쌓기**: 5장 헤드 4개의 질문 쪽지 레시피(4줄 × 16칸)를 위아래로 쌓으면 16줄 × 16칸, 곧 `nn.Linear(16, 16)` 하나가 됩니다.


In [ ]:
query = nn.Linear(16, 16, bias=False)    # 헤드 4개의 질문 쪽지를 Linear 하나로
key = nn.Linear(16, 16, bias=False)
value = nn.Linear(16, 16, bias=False)
wq = []
wk = []
wv = []
for h in mh.heads:                       # 5장 헤드 4개의 레시피를 위아래로 쌓기
    wq.append(h.query.weight)
    wk.append(h.key.weight)
    wv.append(h.value.weight)
with torch.no_grad():
    query.weight.copy_(torch.cat(wq))
    key.weight.copy_(torch.cat(wk))
    value.weight.copy_(torch.cat(wv))
print("query의 가중치 모양:", query.weight.shape)


**② 헤드 세우기**: 글자마다 나온 숫자 16개를 헤드 4개 몫(4개씩)으로 나누고, 헤드를 맨 앞으로 보냅니다.


In [ ]:
T, C = x.shape                           # 글자 9개, 숫자 16개
n_head = 4
hs = C // n_head                         # 헤드 하나의 길이 4
q = query(x).view(T, n_head, hs)         # (9, 4, 4): 글자마다 헤드 4개 몫으로 나누기
k = key(x).view(T, n_head, hs)
v = value(x).view(T, n_head, hs)
q = q.transpose(0, 1)                    # (4, 9, 4): 헤드마다 표 하나
k = k.transpose(0, 1)
v = v.transpose(0, 1)
print("q:", q.shape)


**③ 4장의 네 줄 그대로**: 맨 앞에 헤드 4개가 쌓여 있어도, 뒤의 두 방향(글자, 숫자)으로 계산하는 4장 코드가 그대로 돕니다.


In [ ]:
scores = q @ k.transpose(-2, -1)                          # 4장의 네 줄 그대로
scores = scores / hs ** 0.5
mask = torch.tril(torch.ones(T, T))
scores = scores.masked_fill(mask == 0, float("-inf"))
weights = F.softmax(scores, dim=-1)
out = weights @ v
print("비율 표:", weights.shape)
print("결과:", out.shape)


한꺼번에 계산한 결과의 표 하나하나가 5장 헤드 하나의 결과와 같은지 확인합니다.


In [ ]:
same0 = torch.allclose(out[0], mh.heads[0](x), atol=1e-6)
same3 = torch.allclose(out[3], mh.heads[3](x), atol=1e-6)
print("표 0 = 5장 헤드 0?", same0)
print("표 3 = 5장 헤드 3?", same3)


**④ 이어 붙이기**: 글자를 다시 앞으로 보내고, 헤드 4개의 결과를 이어 붙인 뒤 `proj`로 섞습니다.


In [ ]:
out = out.transpose(0, 1)                # (9, 4, 4): 글자를 다시 앞으로
out = out.reshape(T, C)                  # (9, 16): 헤드 결과 이어 붙이기
same = torch.allclose(mh.proj(out), mh(x), atol=1e-6)
print("5장 MultiHead와 같은가요?", same)


## 8.3 결과가 똑같은지 확인하기

③의 네 줄은 PyTorch가 `F.scaled_dot_product_attention` 함수 하나로 준비해 두었어요. 손으로 한 번 짜 본 것과 같은지 확인합니다.
그다음 패키지의 빠른 GPT가 7장의 기본 GPT와 같은 계산을 하는지 확인해요.

**핵심 코드**: `F.scaled_dot_product_attention(q, k, v, is_causal=True)`


In [ ]:
fast = F.scaled_dot_product_attention(q, k, v, is_causal=True)
same = torch.allclose(weights @ v, fast, atol=1e-6)
print("PyTorch 함수와 같은가요?", same)


In [ ]:
basic = basic.cpu().eval()
fast = GPT(cfg)
copy_basic_to_fast(basic, fast)
fast.eval()

xb, yb = get_batch(data, cfg.block_size, 2)
with torch.no_grad():
    out_basic, _ = basic(xb)
    out_fast, _ = fast(xb)

same = torch.allclose(out_basic, out_fast, atol=1e-4)
print("두 GPT의 결과가 같은가요?", same)


## 8.4 얼마나 빨라졌나 재어 보기

세 가지를 비교합니다. GPU가 없으면 반정밀도는 건너뛰어요.


In [ ]:
t_fast = time_steps(GPT(cfg).to(device), data, n=n)

names = ["기본 GPT", "빠른 GPT"]
times = [t_basic, t_fast]
if device == "cuda":
    t_amp = time_steps(GPT(cfg).to(device), data, n=n, amp=True)
    names.append("빠른 GPT + 반정밀도")
    times.append(t_amp)

for i in range(len(names)):
    t = times[i]
    speedup = t_basic / t
    steps = round(30 * 60 * 1000 / t)
    print(f"{names[i]}: 한 단계 {t:.0f}ms, {speedup:.1f}배, 30분에 약 {steps:,}단계")


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

배치 크기를 바꾸면 한 단계 시간과 **1초에 읽는 글자 수**가 어떻게 달라지는지 보세요. GPU는 일을 한꺼번에 많이 줄수록 효율이 좋아집니다.


In [ ]:
amp = device == "cuda"
for bs in [8, 16, 32, 64]:
    model = GPT(cfg).to(device)
    t = time_steps(model, data, n=n, amp=amp, batch_size=bs)
    chars = round(bs * cfg.block_size / t * 1000)
    print(f"배치 {bs}: 한 단계 {t:.0f}ms, 1초에 {chars:,}글자")


## 실제 LLM에서는

- 실제 LLM은 이 장의 기법에 더해, 16비트 중에서도 범위가 넓은 bfloat16, 더 빠른 어텐션 구현(FlashAttention), 모델을 기계어에 가깝게 바꾸는 컴파일(`torch.compile`), 수천 개 GPU에 나눠 계산하기를 함께 씁니다.
- 원리를 배울 때는 쉬운 코드로, 실제로 학습할 때는 빠른 코드로. 그리고 바꿀 때마다 결과가 같은지 확인합니다. 실제 LLM 개발도 같은 길을 걸어요.
